# Tabulasari Setup

In [2]:
import torch
print(torch.__version__)
print(torch.__file__)

2.2.2
/opt/anaconda3/envs/torch24/lib/python3.11/site-packages/torch/__init__.py


In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

# Use GPU if available
device = 0 if torch.cuda.is_available() else -1

# Load multilingual sentiment model
sentiment_model = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis",
    tokenizer="tabularisai/multilingual-sentiment-analysis",
    device=device,
    truncation=True,
    max_length=512
)

In [ ]:
# Batch sentiment function
def multilingual_sentiment(texts, batch_size=32):
    # Replace non-string values with empty strings
    texts = ["" if not isinstance(t, str) else t for t in texts]

    labels = []
    scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        results = sentiment_model(
            batch,
            truncation=True,
            max_length=512
        )

        labels.extend([r["label"] for r in results])
        scores.extend([r["score"] for r in results])

    return labels, scores

In [ ]:
# read the data in 
teaser = pd.read_csv("./data/teaser_comments.csv")
debut = pd.read_csv("./data/debut_comments.csv")

### Run Model On TEASER Data

In [ ]:
BATCH = 64

labels, scores = multilingual_sentiment(teaser["text"].tolist(), batch_size=BATCH)

teaser["tabulasari_sentiment"] = labels
teaser["tabulasari_confidence"] = scores

print(teaser["tabulasari_sentiment"].value_counts())
print(teaser["tabulasari_confidence"].median())

### Run Model On DEBUT Data

In [ ]:
labels, scores = multilingual_sentiment(debut["text"].tolist(), batch_size=BATCH)

debut["tabulasari_sentiment"] = labels
debut["tabulasari_confidence"] = scores

print(debut["tabulasari_sentiment"].value_counts())
print(debut["tabulasari_confidence"].median())

### Store Dataset with Sentiment Analysis in New CSV File

In [ ]:
# read the teaser sentimental analysis into a new csv
teaser.to_csv('./data/multi_teaser_sentimental_analysis_tabulasari.csv', index = False)

# read the debut sentimental analysis into a new csv
debut.to_csv('./data/multi_debut_sentimental_analysis_tabulasari.csv', index = False)

# XLM Roberta Setup

In [ ]:
# Load XLM-RoBERTa social media sentiment model
sentiment_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length=512
)

### Run on TEASER Data

In [ ]:
BATCH = 64

labels, scores = multilingual_sentiment(teaser["text"].tolist(), batch_size=BATCH)

teaser["xlm_sentiment"] = labels
teaser["xlm_confidence"] = scores

print(teaser["xlm_sentiment"].value_counts())
print(teaser["xlm_confidence"].median())

### Run on DEBUT Data

In [ ]:
labels, scores = multilingual_sentiment(debut["text"].tolist(), batch_size=BATCH)

debut["xlm_sentiment"] = labels
debut["xlm_confidence"] = scores

print(debut["xlm_sentiment"].value_counts())
print(debut["xlm_confidence"].median())

### Store XLM Sentiment in New CSV